# 21SJ-14K (452 JAN) Sales / Stockout / Receipt Analysis

This notebook builds per-JAN charts using CSV files in the current folder.

Charts per JAN:
- Daily sales line
- Stockout periods (shaded spans)
- Receipt timing and quantity (bars + labels)


In [ ]:
from pathlib import Path
import csv
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-whitegrid')
BASE_DIR = Path.cwd()
print('BASE_DIR =', BASE_DIR)


In [ ]:
# ---------- file discovery (ASCII-safe) ----------
all_csv = sorted(BASE_DIR.glob('*.csv'))
if not all_csv:
    raise FileNotFoundError('No CSV files found in current directory.')

# JAN list: name starts with '21SJ-14K JAN'
jan_list_path = next((p for p in all_csv if p.name.startswith('21SJ-14K JAN')), None)
if jan_list_path is None:
    raise FileNotFoundError('JAN list CSV not found (expected prefix: 21SJ-14K JAN).')

# Sales CSV: starts with '21SJ-14K', is CSV, but not JAN list
sales_path = next((p for p in all_csv if p.name.startswith('21SJ-14K') and ('JAN' not in p.name)), None)
if sales_path is None:
    raise FileNotFoundError('Sales CSV not found (expected prefix: 21SJ-14K, excluding JAN list).')

# Detect stockout/receipt files from *_452*.csv candidates
candidates_452 = [p for p in all_csv if '_452' in p.name]

stockout_path = None
receipts_path = None

for p in candidates_452:
    with open(p, 'r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.reader(f))
    if len(rows) < 2:
        continue

    r1 = rows[0]
    r2 = rows[1]

    # stockout: >=4 cols, second and third values look like dates with '-' in row2
    if len(r1) >= 4 and len(r2) >= 4:
        if ('-' in str(r2[1])) and ('-' in str(r2[2])):
            stockout_path = p
            continue

for p in candidates_452:
    with open(p, 'r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.reader(f))
    if len(rows) < 2:
        continue
    r1 = rows[0]
    r2 = rows[1]

    # receipt list from stock ledger: header row + 3 cols, row2 has JAN + date-with-hyphen + qty
    if len(r1) >= 3 and len(r2) >= 3:
        jan_like = bool(re.fullmatch(r'452\d{10}', str(r2[0]).strip()))
        date_like = '-' in str(r2[1])
        qty_like = bool(re.fullmatch(r'-?\d+(\.\d+)?', str(r2[2]).strip()))
        first_row_is_header = not bool(re.fullmatch(r'\d{13}', str(r1[0]).strip()))
        if jan_like and date_like and qty_like and first_row_is_header:
            receipts_path = p
            break

if stockout_path is None:
    raise FileNotFoundError('Stockout CSV (_452) not found.')
if receipts_path is None:
    raise FileNotFoundError('Receipt CSV (_452, header-style) not found.')

print('jan_list_path =', jan_list_path.name)
print('sales_path    =', sales_path.name)
print('stockout_path =', stockout_path.name)
print('receipts_path =', receipts_path.name)


In [ ]:
# ---------- target JAN list ----------
jan_raw = pd.read_csv(jan_list_path, header=None, usecols=[0], names=['jan'], dtype=str, encoding='utf-8-sig')
target_jans = (
    jan_raw['jan']
    .astype(str).str.strip().str.replace('"', '', regex=False)
    .loc[lambda s: s.str.fullmatch(r'452\d{10}', na=False)]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print('target_jan_count =', len(target_jans))
print('target_jan_sample =', target_jans[:10])


In [ ]:
# ---------- load sales ----------
# expected order from your SQL export (headerless)
sales_cols = [
    'year', 'month', 'sales_date', 'customer_code', 'customer_name',
    'jan', 'item_name', 'model', 'color', 'size', 'region', 'sales_qty', 'industry'
]

sales = pd.read_csv(sales_path, header=None, names=sales_cols, dtype=str, encoding='utf-8-sig')
sales['jan'] = sales['jan'].astype(str).str.strip()
sales = sales[sales['jan'].isin(target_jans)].copy()
sales['sales_date'] = pd.to_datetime(sales['sales_date'].astype(str).str.strip(), format='%Y%m%d', errors='coerce')
sales['sales_qty'] = pd.to_numeric(sales['sales_qty'], errors='coerce').fillna(0)
sales = sales.dropna(subset=['sales_date'])

daily_sales = sales.groupby(['jan', 'sales_date'], as_index=False)['sales_qty'].sum()
jan_name_map = (sales.sort_values('sales_date').dropna(subset=['item_name']).drop_duplicates('jan').set_index('jan')['item_name'].to_dict())

print('sales_rows =', len(sales))
print('daily_sales_rows =', len(daily_sales))
print('sales_unique_jan =', daily_sales['jan'].nunique())
print('sales_date_range =', daily_sales['sales_date'].min(), 'to', daily_sales['sales_date'].max())


In [ ]:
# ---------- load stockout ----------
stock_raw = pd.read_csv(stockout_path, encoding='utf-8-sig')
stock_raw = stock_raw.iloc[:, :4].copy()
stock_raw.columns = ['jan', 'stockout_start', 'restock_date', 'days_out']

stock_raw['jan'] = stock_raw['jan'].astype(str).str.strip()
stock_raw = stock_raw[stock_raw['jan'].isin(target_jans)].copy()
stock_raw['stockout_start'] = pd.to_datetime(stock_raw['stockout_start'], errors='coerce')
stock_raw['restock_date'] = pd.to_datetime(stock_raw['restock_date'], errors='coerce')
stock_raw['days_out'] = pd.to_numeric(stock_raw['days_out'], errors='coerce')

stockouts = stock_raw

# ---------- load receipts ----------
rec_raw = pd.read_csv(receipts_path, encoding='utf-8-sig')
rec_raw = rec_raw.iloc[:, :3].copy()
rec_raw.columns = ['jan', 'receipt_date', 'receipt_qty']

rec_raw['jan'] = rec_raw['jan'].astype(str).str.strip()
rec_raw = rec_raw[rec_raw['jan'].isin(target_jans)].copy()
rec_raw['receipt_date'] = pd.to_datetime(rec_raw['receipt_date'], errors='coerce')
rec_raw['receipt_qty'] = pd.to_numeric(rec_raw['receipt_qty'], errors='coerce').fillna(0)
rec_raw = rec_raw.dropna(subset=['receipt_date'])

receipts_daily = rec_raw.groupby(['jan', 'receipt_date'], as_index=False)['receipt_qty'].sum()

print('stockout_rows =', len(stockouts), '/ unique_jan =', stockouts['jan'].nunique())
print('receipts_rows =', len(receipts_daily), '/ unique_jan =', receipts_daily['jan'].nunique())


In [ ]:
# ---------- data coverage summary ----------
jan_set = set(target_jans)
jan_sales = set(daily_sales['jan'].unique())
jan_stock = set(stockouts['jan'].unique())
jan_receipt = set(receipts_daily['jan'].unique())

summary = pd.DataFrame({'jan': sorted(jan_set)})
summary['has_sales'] = summary['jan'].isin(jan_sales)
summary['has_stockout'] = summary['jan'].isin(jan_stock)
summary['has_receipt'] = summary['jan'].isin(jan_receipt)

print(summary.head(20).to_string(index=False))
print('no_sales_jan_count =', (~summary['has_sales']).sum())
print('no_stockout_jan_count =', (~summary['has_stockout']).sum())
print('no_receipt_jan_count =', (~summary['has_receipt']).sum())


In [ ]:
def plot_jan(jan_code: str, save_path=None, show=True):
    js = daily_sales[daily_sales['jan'] == jan_code].copy()
    jr = receipts_daily[receipts_daily['jan'] == jan_code].copy()
    jz = stockouts[stockouts['jan'] == jan_code].copy()

    if js.empty and jr.empty and jz.empty:
        return None

    dates = []
    if not js.empty:
        dates += [js['sales_date'].min(), js['sales_date'].max()]
    if not jr.empty:
        dates += [jr['receipt_date'].min(), jr['receipt_date'].max()]
    if not jz.empty:
        dates += [jz['stockout_start'].min(), jz['restock_date'].max()]

    dmin, dmax = min(dates), max(dates)
    if pd.isna(dmin) or pd.isna(dmax):
        return None

    idx = pd.date_range(dmin, dmax, freq='D')
    if not js.empty:
        y = js.set_index('sales_date')['sales_qty'].reindex(idx, fill_value=0)
    else:
        y = pd.Series(0, index=idx, dtype=float)

    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(idx, y.values, color='tab:blue', linewidth=1.2, label='Daily Sales Qty')
    ax.set_ylabel('Sales Qty')
    ax.set_ylim(bottom=0)

    first_span = True
    for _, row in jz.iterrows():
        s, e = row['stockout_start'], row['restock_date']
        if pd.notna(s) and pd.notna(e) and e >= s:
            ax.axvspan(s, e, color='tab:red', alpha=0.15, label='Stockout Period' if first_span else None)
            first_span = False

    ax2 = ax.twinx()
    if not jr.empty:
        ax2.bar(jr['receipt_date'], jr['receipt_qty'], width=2.0, color='tab:green', alpha=0.35, label='Receipt Qty')
        for _, row in jr.iterrows():
            q = row['receipt_qty']
            if pd.notna(q) and q != 0:
                txt = f'{int(q)}' if float(q).is_integer() else f'{q:.1f}'
                ax2.text(row['receipt_date'], q, txt, fontsize=7, rotation=90, ha='center', va='bottom', color='tab:green')
    ax2.set_ylabel('Receipt Qty')
    ax2.set_ylim(bottom=0)

    ax.set_title(jan_code)

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', frameon=True)

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=140, bbox_inches='tight')

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig


In [ ]:
# Save chart for every target JAN
out_dir = BASE_DIR / 'charts_jan_sales_stockout_receipt'
out_dir.mkdir(exist_ok=True)

created = 0
for jan in target_jans:
    fig = plot_jan(jan, save_path=out_dir / f'{jan}.png', show=False)
    if fig is not None:
        created += 1

print('saved_chart_count =', created)
print('output_dir =', out_dir)


In [ ]:
# Preview first 3 JANs in notebook output
for jan in target_jans[:3]:
    _ = plot_jan(jan, show=True)
